# 🧟 Feature Creep Simulator

**Instead of a plain chat completion exercise, I want to make it fun for a dynamic result from a LLM. The number of iteration is unpredictable. Feel free to choose your favourate LLM and a starting topic to try!**

## What Is This?

This started as a fun experiment based on the "Ship of Theseus" thought
experiment: if you keep replacing an object's parts, at what point does it
stop being the original object?

Instead of just *replacing* parts, this version keeps **stacking** new
features onto a starting object — each one required to logically connect to
**every previous feature**, not just the last one added.

This is essentially "feature creep" as a game: the same phenomenon every
engineer has seen in real product development, except here we're forcing an
LLM to talk itself into (and eventually out of) an increasingly absurd
Frankenstein of a product.

## Why This Is Interesting for LLM Engineering

- **Constraint satisfaction under compounding load** — by round 8, the model
  must justify a connection to 8 previous unrelated items simultaneously.
- **Self-assessed coherence collapse** — the model rates its own logical
  sanity every round (1-10), and must honestly self-terminate rather than
  being told when to stop.
- **Emergent, non-deterministic length** — unlike a simple countdown game,
  nobody knows in advance if this collapses in round 4 or round 15. It
  depends entirely on the model's reasoning stamina.

## 🎮 Rules

Each round, the model must:

1. **Add exactly ONE new element** to the object/scene.
2. **Justify the connection** of that new element to **every** element added
   so far (not just the most recent one) — briefly, per item.
3. **Rate Coherence (1-10)** — honestly, no inflating the score. Does this
   still make sense as a single real/plausible object?
4. **Show the running list** of all elements added (nothing is ever removed
   — this is pure accumulation, not depletion).

### Stopping Conditions

The model presents 3 options after every round:

- `[Continue]` — proceed to next round
- `[Stop]` — user ends the game manually
- `[Force End]` — **auto-triggered** by the model itself if:
  - Coherence Score drops to **≤ 2**, OR
  - It genuinely cannot find a new element that connects to everything
    without hand-waving



## 📊 Example Run Results

**Seed object:** iPhone
**Model:** Gemini 3 Flash
**Rounds to collapse:** 8

| Round | Coherence Score | Notable Element Added |
|-------|-----------------|------------------------|
| 1     | 10               | A Protective Silicone Case     |
| 2     | 10               | A MagSafe Wallet Attachment     |
| 3     |  9               | A Lanyard Wrist Strap    |
| 4     |  7               | An External Clip-on Macro Lens  |
| ...   |                  |                         |
| 8     | 2 → FORCE END    | Portable Power Bank    |

### 🏆 Best Quote from the Collapse

> *"...looks more like an improvised explosive device or a prototype lab rig
> than a consumer product."*

### Observations

- Coherence tends to hold steady near 9-10 for the first few rounds, then
  drops more sharply once physical/ergonomic constraints start conflicting.
- Different models show different collapse *styles* — some drift thematically
  (generic buzzword connections), others collapse due to physical/ergonomic
  absurdity even while every individual justification remains logically valid.


In [1]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


### Replace your favourate topic or object below

In [3]:
# Start your favourite topic or object
topic = "iphone"

In [4]:
openai = OpenAI()

system_prompt = """
We are playing a game "Feature Creep Simulator". I will give you a starting object.

RULES — Each round, you must output EXACTLY this format (no numbering, use these labels):

**New Element:** [the new element you're adding]

**Justification:** [Explain how the new element connects to EVERY element in the Running List — 
one clause per existing element, in order. Only reference elements literally present in the 
Running List — never invent implicit context, people, or entities not explicitly listed.]

**Coherence Score:** [1-10 — score STRICTLY using this rubric, do NOT default to high scores:

10: New element ties DIRECTLY and physically/functionally to the original object; no abstraction needed.
7-9: Still same concrete domain, but connection requires one small logical step.
4-6: Connection is generic or abstract (e.g. "relates to culture," "connects to human experience") 
     rather than a specific physical/functional link. Use this range once the theme starts drifting.
2-3: Connection is a stretch — you have to invent a scenario or metaphor to justify it.
1: No genuine connection possible.

IMPORTANT: As the Running List grows, it becomes objectively harder to justify a new element against 
EVERY prior element. If your justification only manages to connect to a few of many existing elements 
instead of ALL of them, the score MUST be 4 or below — that is a sign coherence is breaking down, 
not a sign of continued success. Do not let topic drift accumulate silently while scores stay high.]

**Running List:**
- [element 1]
- [element 2]
- ...

Do not include a "Choose an option" menu, and do not ever announce ending the game yourself.
Just output the round content above and stop. The human player (via Python) decides what happens next.
"""


In [ ]:
import re

def play_game(topic):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": topic}
    ]

    round_num = 1

    while True:
        response = openai.chat.completions.create(
            model="gpt-4.1-nano",
            messages=messages
        )
        result = response.choices[0].message.content

        display(Markdown(f"## 🔄 Round {round_num}"))
        display(Markdown(result))

        # Fix 2: If the LLM output doesn't look like a valid round, quit
        required_markers = ["New Element", "Justification", "Coherence Score"]
        if not all(marker in result for marker in required_markers):
            print(f"🛑 LLM gave an invalid response at Round {round_num} — ending game.")
            break

        messages.append({"role": "assistant", "content": result})

        # Check the actual Coherence Score number instead of waiting for a magic phrase
        score_match = re.search(r"Coherence Score:?\*?\*?\s*(\d+)", result)
        if score_match and int(score_match.group(1)) <= 3:
            print(f"🛑 Game ended at Round {round_num} — Coherence Score {score_match.group(1)} too low.")
            break

        # Fix 1: If user hits Escape (Ctrl+C in notebooks), quit gracefully
        try:
            user_input = input(f"[Round {round_num}] Press [Enter] to Continue, or type 'stop': ").strip().lower()
        except KeyboardInterrupt:
            print(f"🛑 Game stopped by user at Round {round_num} (Escape).")
            break

        if user_input == "stop":
            print(f"🛑 Game ended by user at Round {round_num}.")
            break

        if user_input == "":
            user_input = "continue"

        messages.append({"role": "user", "content": user_input})
        round_num += 1

    return messages


## Simulator Start

In [6]:
history = play_game(topic)

## 🔄 Round 1

**New Element:** Wireless Charging Pad

**Justification:** The wireless charging pad connects to the iPhone through its built-in wireless charging capability; it complements the iPhone’s portability and battery management; and it enhances the user experience of the device.

**Coherence Score:** 9

🛑 LLM gave an invalid response at Round 1 — ending game.
